# RetailIQ 360° — EDA Nivel 2: Exploración por grupo funcional

**Objetivo:** Verificar que los datos son coherentes *entre sí* — no solo dentro de cada archivo.

**Pregunta que responde cada bloque:**
- Bloque 1: ¿El núcleo transaccional (orders + items + products) hace un join limpio?
- Bloque 2: ¿El contexto económico (IPC + tipo de cambio) cubre todo el período de Olist?
- Bloque 3: ¿Los benchmarks (Superstore + CACE) son compatibles con las categorías del modelo?
- Bloque 4: ¿Cómo quedaría la tabla base de FactVentas con todos los datos unidos?

**Prerequisito:** haber corrido el EDA Nivel 1 (`02_EDA_nivel_1.ipynb`) al menos una vez.

## BLOQUE 0 — Configuración y carga de datos

Cargamos todos los archivos desde cero. No dependemos del estado del kernel del nivel 1.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

RAW_DIR  = '../datos/01_raw'
PROC_DIR = '../datos/04_procesados'
CACE_DIR = '../datos/02_cace_benchmarks'

# ── Olist ────────────────────────────────────────────────────
df_orders      = pd.read_csv(f'{RAW_DIR}/001_olist_orders_dataset.csv')
df_items       = pd.read_csv(f'{RAW_DIR}/002_olist_order_items_dataset.csv')
df_products    = pd.read_csv(f'{RAW_DIR}/004_olist_products_dataset.csv')
df_translation = pd.read_csv(f'{RAW_DIR}/005_product_category_name_translation.csv')

# ── Superstore ───────────────────────────────────────────────
df_super = pd.read_csv(f'{RAW_DIR}/006_Sample - Superstore.csv', encoding='latin-1')

# ── Contexto económico ───────────────────────────────────────
df_ipc = pd.read_csv(f'{PROC_DIR}/dim_inflacion_ipc.csv', parse_dates=['fecha'])
df_tc  = pd.read_csv(f'{RAW_DIR}/008_tipos-de-cambio-historicos.csv',
                     parse_dates=['indice_tiempo'])

# Convertir fechas clave
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])

# Solo órdenes delivered — las que entran a FactVentas
orders_delivered = df_orders[df_orders['order_status'] == 'delivered'].copy()

print('✓ Todos los archivos cargados')
print(f'\nPeríodo de Olist (delivered):')
print(f'  Desde: {orders_delivered["order_purchase_timestamp"].min().date()}')
print(f'  Hasta: {orders_delivered["order_purchase_timestamp"].max().date()}')
print(f'  Total órdenes delivered: {len(orders_delivered):,}')

✓ Todos los archivos cargados

Período de Olist (delivered):
  Desde: 2016-09-15
  Hasta: 2018-08-29
  Total órdenes delivered: 96,478


---
## BLOQUE 1 — Núcleo transaccional (FactVentas)

Verificamos que los tres archivos centrales hacen un join limpio.
La cadena es: `orders → items → products`

Si hay órdenes sin ítems o ítems sin producto, perdemos filas al hacer el join en el ETL.

In [2]:
# ── 1.1 Integridad referencial: orders ↔ items ───────────────
orders_con_items    = set(df_items['order_id'].unique())
orders_delivered_ids = set(orders_delivered['order_id'].unique())

sin_items        = orders_delivered_ids - orders_con_items
items_sin_order  = set(df_items['order_id'].unique()) - set(df_orders['order_id'].unique())

print('── ORDERS ↔ ITEMS ──────────────────────────────────────')
print(f'Órdenes delivered:              {len(orders_delivered_ids):>8,}')
print(f'Órdenes delivered SIN ítems:    {len(sin_items):>8,}  {"✓" if len(sin_items) == 0 else "⚠ revisar"}')
print(f'Ítems con order_id inexistente: {len(items_sin_order):>8,}  {"✓" if len(items_sin_order) == 0 else "⚠ revisar"}')

── ORDERS ↔ ITEMS ──────────────────────────────────────
Órdenes delivered:                96,478
Órdenes delivered SIN ítems:           0  ✓
Ítems con order_id inexistente:        0  ✓


In [3]:
# ── 1.2 Integridad referencial: items ↔ products ─────────────
products_ids      = set(df_products['product_id'].unique())
items_product_ids = set(df_items['product_id'].unique())

items_sin_producto = items_product_ids - products_ids

print('── ITEMS ↔ PRODUCTS ────────────────────────────────────')
print(f'product_id únicos en items:     {len(items_product_ids):>8,}')
print(f'product_id únicos en products:  {len(products_ids):>8,}')
print(f'product_id en items sin ficha:  {len(items_sin_producto):>8,}  {"✓" if len(items_sin_producto) == 0 else "⚠ revisar"}')

if items_sin_producto:
    filas_afectadas = df_items[df_items['product_id'].isin(items_sin_producto)].shape[0]
    print(f'  → Filas de items afectadas:   {filas_afectadas:>8,}')
    print(f'  → ETL: usar left join, categoría = "Sin ficha" para esos productos')

── ITEMS ↔ PRODUCTS ────────────────────────────────────
product_id únicos en items:       32,951
product_id únicos en products:    32,951
product_id en items sin ficha:         0  ✓


In [4]:
# ── 1.3 Distribución temporal de ventas ──────────────────────
orders_delivered['anio_mes'] = orders_delivered['order_purchase_timestamp'].dt.to_period('M')
ventas_por_mes = orders_delivered.groupby('anio_mes').size()

print('── VENTAS POR MES (órdenes delivered) ──────────────────')
print(ventas_por_mes.to_string())
print(f'\nMes con más ventas:   {ventas_por_mes.idxmax()} ({ventas_por_mes.max():,} órdenes)')
print(f'Mes con menos ventas: {ventas_por_mes.idxmin()} ({ventas_por_mes.min():,} órdenes)')

── VENTAS POR MES (órdenes delivered) ──────────────────
anio_mes
2016-09       1
2016-10     265
2016-12       1
2017-01     750
2017-02    1653
2017-03    2546
2017-04    2303
2017-05    3546
2017-06    3135
2017-07    3872
2017-08    4193
2017-09    4150
2017-10    4478
2017-11    7289
2017-12    5513
2018-01    7069
2018-02    6555
2018-03    7003
2018-04    6798
2018-05    6749
2018-06    6099
2018-07    6159
2018-08    6351
Freq: M

Mes con más ventas:   2017-11 (7,289 órdenes)
Mes con menos ventas: 2016-09 (1 órdenes)


In [5]:
# ── 1.4 Ticket promedio y distribución de precios ────────────
fact_base = orders_delivered[['order_id']].merge(
    df_items[['order_id', 'price', 'freight_value']],
    on='order_id', how='inner'
)

ticket_por_orden = fact_base.groupby('order_id')['price'].sum()

print('── TICKET POR ORDEN (BRL) ──────────────────────────────')
print(f'Órdenes con precio:    {len(ticket_por_orden):>8,}')
print(f'Ticket promedio:       R$ {ticket_por_orden.mean():>10,.2f}')
print(f'Ticket mediano:        R$ {ticket_por_orden.median():>10,.2f}')
print(f'Ticket mínimo:         R$ {ticket_por_orden.min():>10,.2f}')
print(f'Ticket máximo:         R$ {ticket_por_orden.max():>10,.2f}')
print(f'\nDistribución por rango de precio (BRL):')
bins   = [0, 50, 100, 200, 500, 1000, float('inf')]
labels = ['0-50', '50-100', '100-200', '200-500', '500-1000', '1000+']
rangos = pd.cut(ticket_por_orden, bins=bins, labels=labels)
print(rangos.value_counts().sort_index().to_frame('órdenes'))

── TICKET POR ORDEN (BRL) ──────────────────────────────
Órdenes con precio:      96,478
Ticket promedio:       R$     137.04
Ticket mediano:        R$      86.57
Ticket mínimo:         R$       0.85
Ticket máximo:         R$  13,440.00

Distribución por rango de precio (BRL):
          órdenes
price            
0-50        29045
50-100      27606
100-200     25289
200-500     11061
500-1000     2572
1000+         905


---
## BLOQUE 2 — Cobertura del contexto económico

Verificamos que IPC y tipo de cambio cubren **todos los meses** del período de Olist.
Si hay gaps, definimos cómo manejarlos en el ETL.

In [6]:
# ── 2.1 Cobertura del IPC ────────────────────────────────────
# El IPC del INDEC empieza en enero 2017 pero Olist arranca en sep-2016
meses_olist = pd.period_range(
    start=orders_delivered['order_purchase_timestamp'].min().to_period('M'),
    end=orders_delivered['order_purchase_timestamp'].max().to_period('M'),
    freq='M'
)

meses_ipc = pd.PeriodIndex(df_ipc['fecha'].dt.to_period('M'))

meses_sin_ipc = [m for m in meses_olist if m not in meses_ipc]

print('── COBERTURA IPC vs OLIST ──────────────────────────────')
print(f'Meses en Olist (delivered):  {len(meses_olist):>4}')
print(f'Meses con datos IPC:         {len(meses_ipc):>4}')
print(f'Meses SIN IPC:               {len(meses_sin_ipc):>4}  {"✓" if len(meses_sin_ipc) == 0 else "⚠ ver abajo"}')

if meses_sin_ipc:
    print(f'\nMeses sin cobertura IPC:')
    for m in meses_sin_ipc:
        n = orders_delivered[
            orders_delivered['order_purchase_timestamp'].dt.to_period('M') == m
        ].shape[0]
        print(f'  {m}  →  {n:,} órdenes delivered')
    print(f'\n→ Decisión ETL: para esos meses ipc = 0 (sin ajuste) o imputar con primer valor disponible')

── COBERTURA IPC vs OLIST ──────────────────────────────
Meses en Olist (delivered):    24
Meses con datos IPC:          111
Meses SIN IPC:                  4  ⚠ ver abajo

Meses sin cobertura IPC:
  2016-09  →  1 órdenes delivered
  2016-10  →  265 órdenes delivered
  2016-11  →  0 órdenes delivered
  2016-12  →  1 órdenes delivered

→ Decisión ETL: para esos meses ipc = 0 (sin ajuste) o imputar con primer valor disponible


In [7]:
# ── 2.2 Cobertura del tipo de cambio ─────────────────────────
col_tc = 'dolar_estadounidense'
df_tc_valido = df_tc[['indice_tiempo', col_tc]].dropna().copy()
df_tc_valido.columns = ['fecha', 'usd_ars']

fechas_olist = orders_delivered['order_purchase_timestamp'].dt.date
fechas_tc    = set(df_tc_valido['fecha'].dt.date)

fechas_sin_tc = [f for f in fechas_olist.unique() if f not in fechas_tc]

print('── COBERTURA TIPO DE CAMBIO vs OLIST ───────────────────')
print(f'Días únicos con órdenes Olist: {fechas_olist.nunique():>6,}')
print(f'Días con tipo de cambio:       {len(fechas_tc):>6,}')
print(f'Días de Olist SIN tipo cambio: {len(fechas_sin_tc):>6,}  {"✓" if len(fechas_sin_tc) == 0 else "⚠ ver abajo"}')

if fechas_sin_tc:
    print(f'\nPrimeros 10 días sin cobertura:')
    for f in sorted(fechas_sin_tc)[:10]:
        print(f'  {f}')
    print(f'\n→ Decisión ETL: forward fill — usar el último valor conocido')
    print(f'  Los fines de semana y feriados no tienen cotización, es normal')

── COBERTURA TIPO DE CAMBIO vs OLIST ───────────────────
Días únicos con órdenes Olist:    612
Días con tipo de cambio:       12,488
Días de Olist SIN tipo cambio:      0  ✓


---
## BLOQUE 3 — Benchmarks (Superstore + CACE)

Verificamos que las categorías de los benchmarks pueden mapearse
a las categorías del modelo (que vienen de Olist).

In [8]:
# ── 3.1 Top categorías Olist vs categorías Superstore ────────
cats_olist = df_items.merge(
    df_products[['product_id', 'product_category_name']], on='product_id', how='left'
).merge(
    df_translation, on='product_category_name', how='left'
)

print('── TOP 10 CATEGORÍAS OLIST (por ítems vendidos) ────────')
print(cats_olist['product_category_name_english'].value_counts().head(10).to_frame('ítems'))

print('\n── CATEGORÍAS SUPERSTORE ───────────────────────────────')
print(df_super[['Category', 'Sub-Category']].drop_duplicates().sort_values('Category').to_string(index=False))

print('\n→ Mapeo sugerido Category Superstore → categoría Olist para FactPreciosComp:')
mapeo = [
    ('Technology → Phones',             'telefonia'),
    ('Technology → Computers/Acc.',     'informatica_acessorios'),
    ('Furniture → Chairs/Bookcases',    'moveis_decoracao'),
    ('Office Supplies → resto',         'utilidades_domesticas'),
]
for sup, olis in mapeo:
    print(f'  {sup:<40} → {olis}')

── TOP 10 CATEGORÍAS OLIST (por ítems vendidos) ────────
                               ítems
product_category_name_english       
bed_bath_table                 11115
health_beauty                   9670
sports_leisure                  8641
furniture_decor                 8334
computers_accessories           7827
housewares                      6964
watches_gifts                   5991
telephony                       4545
garden_tools                    4347
auto                            4235

── CATEGORÍAS SUPERSTORE ───────────────────────────────
       Category Sub-Category
      Furniture    Bookcases
      Furniture       Chairs
      Furniture       Tables
      Furniture  Furnishings
Office Supplies     Supplies
Office Supplies    Fasteners
Office Supplies    Envelopes
Office Supplies        Paper
Office Supplies   Appliances
Office Supplies      Binders
Office Supplies          Art
Office Supplies      Storage
Office Supplies       Labels
     Technology       Phones
     T

In [9]:
# ── 3.2 Vista rápida de los archivos CACE ────────────────────
print('── ARCHIVOS CACE DISPONIBLES ───────────────────────────')
for f in sorted(os.listdir(CACE_DIR)):
    if f.endswith('.csv') and f != 'cace_00_README.csv':
        df_tmp = pd.read_csv(f'{CACE_DIR}/{f}')
        print(f'  {f:<50} {df_tmp.shape[0]:>4} filas  {df_tmp.shape[1]:>3} cols')

print('\n── KPIs MACRO CACE ─────────────────────────────────────')
df_kpis = pd.read_csv(f'{CACE_DIR}/cace_01_kpis_macro.csv')
print(df_kpis.to_string(index=False))

── ARCHIVOS CACE DISPONIBLES ───────────────────────────
  cace_01_kpis_macro.csv                               49 filas    7 cols
  cace_02_categorias_rubros.csv                        13 filas   13 cols
  cace_03a_conversion_por_categoria.csv                 5 filas    6 cols
  cace_03b_ranking_demanda.csv                         10 filas    6 cols
  cace_04a_medios_pago_oferta.csv                       8 filas   10 cols
  cace_04b_financiamiento_cuotas.csv                    5 filas    6 cols
  cace_05a_logistica_tipo_entrega.csv                   5 filas    7 cols
  cace_05b_plazos_entrega.csv                           6 filas    9 cols
  cace_06a_distribucion_regional.csv                    7 filas    8 cols
  cace_06b_perfil_comprador.csv                        12 filas    5 cols
  cace_07_canal_online_por_categoria.csv                9 filas    6 cols

── KPIs MACRO CACE ─────────────────────────────────────
                                     indicador  anio periodo       valo

---
## BLOQUE 4 — Vista previa del join completo (base de FactVentas)

Unimos todos los datasets para ver cómo quedaría la tabla base antes del ETL.
Esto confirma que los joins funcionan y que no perdemos filas inesperadamente.

In [10]:
# ── 4.1 Construir la base de FactVentas paso a paso ──────────
print('── CONSTRUCCIÓN PASO A PASO ────────────────────────────')

# Paso 1: solo órdenes delivered
base = orders_delivered[['order_id', 'order_purchase_timestamp']].copy()
print(f'[1] orders delivered:                    {len(base):>8,} filas')

# Paso 2: join con items
base = base.merge(
    df_items[['order_id', 'product_id', 'seller_id', 'price', 'freight_value']],
    on='order_id', how='inner'
)
print(f'[2] + items (inner join):                {len(base):>8,} filas')

# Paso 3: join con products
base = base.merge(
    df_products[['product_id', 'product_category_name']],
    on='product_id', how='left'
)
print(f'[3] + products (left join):              {len(base):>8,} filas')

# Paso 4: traducción de categorías
base = base.merge(df_translation, on='product_category_name', how='left')
base['product_category_name_english'] = base['product_category_name_english'].fillna('other')
print(f'[4] + traducción categorías (left join): {len(base):>8,} filas')

# Paso 5: columnas de fecha para joins futuros con IPC y TC
base['anio']     = base['order_purchase_timestamp'].dt.year
base['mes']      = base['order_purchase_timestamp'].dt.month
base['fecha_dia'] = base['order_purchase_timestamp'].dt.normalize()

print(f'\n✓ Tabla base lista: {len(base):,} filas × {len(base.columns)} columnas')

── CONSTRUCCIÓN PASO A PASO ────────────────────────────
[1] orders delivered:                      96,478 filas
[2] + items (inner join):                 110,197 filas
[3] + products (left join):               110,197 filas
[4] + traducción categorías (left join):  110,197 filas

✓ Tabla base lista: 110,197 filas × 11 columnas


In [11]:
# ── 4.2 Resumen final ────────────────────────────────────────
print('── PRIMERAS 5 FILAS ────────────────────────────────────')
cols_mostrar = ['order_id', 'order_purchase_timestamp',
                'product_category_name_english', 'price', 'freight_value', 'anio', 'mes']
print(base[cols_mostrar].head().to_string(index=False))

print('\n── RESUMEN FINAL ───────────────────────────────────────')
print(f'Total ítems en FactVentas:   {len(base):>10,}')
print(f'Órdenes únicas:              {base["order_id"].nunique():>10,}')
print(f'Facturación total (BRL):  R$ {base["price"].sum():>14,.2f}')
print(f'Categorías distintas:        {base["product_category_name_english"].nunique():>10,}')
print(f'Período:  {base["order_purchase_timestamp"].min().date()} → {base["order_purchase_timestamp"].max().date()}')

print('\n══════════════════════════════════════════════════════')
print('EDA NIVEL 2 COMPLETO')
print('Próximo paso: ETL — aplicar conversión BRL→ARS')
print('y ajuste por inflación (IPC INDEC) para obtener precios reales')
print('══════════════════════════════════════════════════════')

── PRIMERAS 5 FILAS ────────────────────────────────────
                        order_id order_purchase_timestamp product_category_name_english  price  freight_value  anio  mes
e481f51cbdc54678b7cc49136f2d6af7      2017-10-02 10:56:33                    housewares  29.99           8.72  2017   10
53cdb2fc8bc7dce0b6741e2150273451      2018-07-24 20:41:37                     perfumery 118.70          22.76  2018    7
47770eb9100c2d0c44946d9cf07ec65d      2018-08-08 08:38:49                          auto 159.90          19.22  2018    8
949d5b44dbf5de918fe9c16f97b45f8a      2017-11-18 19:28:06                      pet_shop  45.00          27.20  2017   11
ad21c59c0840e6cb83a9ceb5573f8159      2018-02-13 21:18:39                    stationery  19.90           8.72  2018    2

── RESUMEN FINAL ───────────────────────────────────────
Total ítems en FactVentas:      110,197
Órdenes únicas:                  96,478
Facturación total (BRL):  R$  13,221,498.11
Categorías distintas:              